Installing necessary dependencies

In [ ]:
!pip install qdrant-client
!pip install open-clip-torch
!pip install python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.7/327.7 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 80.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.

Importing necessary libraries

In [ ]:
import sqlite3
import os
from transformers import AutoProcessor, AutoModelForVision2Seq
from PIL import Image
import torch
import re
import pandas as pd
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct
import requests
from dotenv import load_dotenv

Database setup

In [ ]:
# Database setup
DB_PATH = "mllm_experiments.db"

def create_db():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS experiment_results_test2 (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            image_name TEXT,
            question TEXT,
            context TEXT,
            object TEXT,
            grounding TEXT,
            image_tag TEXT,
            date TEXT,
            model_response TEXT
        )
    """)
    conn.commit()
    conn.close()

create_db()


#Store Results in Database
def store_experiment_result(image_name, tags, response):
    """Safe database insertion with all required fields"""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("""
        INSERT INTO experiment_results_test2
        (image_name, question, context, object, grounding, image_tag, date, model_response)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        image_name,
        tags["question"],
        tags["context"],
        tags["object"],
        tags["grounding"],
        tags["image"],
        tags.get("date", "2023-01-01"),
        response
    ))

    conn.commit()
    conn.close()
    print(f"Stored result for {image_name}")

Clean LLM Output

In [ ]:

def clean_response(response):
    """Clean the response by removing unnecessary tags and repetitive text."""
    # Remove unwanted tags and noise
    unwanted_patterns = [
        r"<image>.*?</image>",  # Remove <image> tags and their content
        r"<grounding>.*?</grounding>",  # Remove <grounding> tags
        r"<context>.*?</context>",  # Remove <context> tags
        r"<image>.*?</image>",  # Remove <image> tags
        r"<object>.*?</object>",  # Remove <object> tags
        r"What can you see in this Sentinel-2A satellite image of.*?\?",  # Remove repetitive question
    ]

    for pattern in unwanted_patterns:
        response = re.sub(pattern, "", response, flags=re.DOTALL)

    # Remove extra spaces and newlines
    response = " ".join(response.split())

    return response.strip()


In [ ]:
# Load Model (Kosmos-2) and Setup
#ran over 10 images in the drive

# Disable oneDNN optimizations if needed
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

# Set the path where your data is stored
datapath = r"/content"  # <-- Update this path to where your images are stored

# Load the Kosmos-2 model (this will dow bnload ~7GB on first run)
model_source = "microsoft/kosmos-2-patch14-224"
processor = AutoProcessor.from_pretrained(model_source)
model = AutoModelForVision2Seq.from_pretrained(model_source)

if torch.cuda.is_available():
    model = model.to("cuda")
    print("Using CUDA")

# Generate prompts for multiple images

def generate_prompt(tags):

    """
    Generates a structured paratactic prompt.

    :param question: The main question.
    :param tags: Keyword arguments representing tag-value pairs.
    :return: Formatted prompt string.
    """
    prompt = f"""<grounding> {tags['grounding']} </grounding>
    <image> {tags['image']} </image>
    <context> {tags['context']} </context>
    <object> {tags['object']} </object>
    {tags['question']}"""
    return prompt

# List of input images to process
image_files = [
    "image_1.png",
    "image_2.png"

]


# Metadata Extraction from Filenames
def metadata(filename):
  parts = filename.split("_")
  if len(parts)>0:
    location = parts[0]
  else:
    location = ""

  for i in parts:
    if len(i) == 10 and i[4] == "-" and i[7] == "-":
      date = i
      break
    else:
      date = ""

  if "fmr" in filename.lower():
    additional_info = "Ferrous Mineral Ratio (FMR)"
  elif "ndvi" in filename.lower():
    additional_info = "Normalized Difference Vegetation Index (NDVI)"
  else:
    additional_info = "Unknown"

  return {"location": location, "date": date, "additional_info": additional_info}


# Process Multiple Images
for img_file in image_files:
    try:
        # Extract metadata
        meta_data = metadata(img_file)
        location = meta_data["location"]
        date = meta_data["date"]
        info = meta_data["additional_info"]

        # Generate question

        question = f"What can you see in this Sentinel-2A satellite image of {location}?"

        # Prepare tags
        tags = {
            "question": question,
            "grounding": f"Sentinel-2A satellite data.",
            "context": f"This is a satellite image derived from Sentinel-2A of {location} captured on {date}.",
            "image": f"Analyze the {info} in the image.",
            "object": f"Identify key objects in the image related to {info}.",
            "date": date
        }


        # Generate prompt
        prompt = generate_prompt(tags)


        # Process image
        image = Image.open(os.path.join(datapath, img_file))
        inputs = processor(
            text=prompt,
            images=image,
            return_tensors="pt",
            padding=True
        )

        if torch.cuda.is_available():
            inputs = {k: v.to("cuda") for k, v in inputs.items()}

        # Generate response
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                min_length=8,
                num_beams=5,
                max_new_tokens=256,
                length_penalty=1.0,
                early_stopping = True
            )

        response = processor.batch_decode(outputs, skip_special_tokens=True)[0]

        # Clean the response
        cleaned_response = clean_response(response)

        # Store results
        store_experiment_result(img_file, tags, cleaned_response)
        print(f"Processed {img_file}\nResponse: {cleaned_response}\n{'='*80}")

    except Exception as e:
        print(f"Error processing {img_file}: {str(e)}")


# Export All Results to CSV
def export_to_csv():
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql_query("SELECT * FROM experiment_results_test2", conn)
    df.to_csv("experiment_results_test.csv", index=False)
    conn.close()
    print("Data exported to experiment_results_test2.csv")

export_to_csv()




preprocessor_config.json:   0%|          | 0.00/534 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


tokenizer_config.json:   0%|          | 0.00/191k [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.70M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.45k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/6.66G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Stored result for image_1.png
Processed image_1.png
Response: In the image, you can see a large city with a river running through it. The city appears to be located near a body of water, such as a river, lake, or ocean. Additionally, there are numerous buildings scattered throughout the city, indicating that it is a densely populated urban area. The presence of the river suggests that the city is situated near a significant water source, possibly a river or a lake. The buildings in the city may be residential, commercial, or industrial structures. Overall, the image portrays a bustling urban environment with a mix of residential and commercial structures.
Stored result for image_2.png
Processed image_2.png
Response: In the image, you can see a large city with a densely populated urban area. The city appears to be densely populated, with numerous buildings, roads, and parks. There are also a few trees visible in the city, adding to the urban environment. The image also shows a river run

In [ ]:
# Embed Captions with SentenceTransformer + Qdrant Search

# Load environment variables from the .env file
load_dotenv()

# Get API key securely
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")

if not QDRANT_API_KEY:
    raise EnvironmentError("QDRANT_API_KEY not found. Please set it in the .env file.")

# Create Qdrant client
client = QdrantClient(
    url="https://598f0792-a4c3-422d-b44b-a983e3453c52.us-east4-0.gcp.cloud.qdrant.io:6333",
    api_key=QDRANT_API_KEY
)

collection_name = "test_vector_collection"


In [ ]:
client.recreate_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)


<ipython-input-7-5b25a3e62f1b>:1: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


True

In [ ]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
documents = [
    {"id": 1, "image_name": "image_1.png", "caption": "In the image, you can see a large city with a river running through it. The city appears to be located near a body of water, such as a river, lake, or ocean. Additionally, there are numerous buildings scattered throughout the city, indicating that it is a densely populated urban area. The presence of the river suggests that the city is situated near a significant water source, possibly a river or a lake. The buildings in the city may be residential, commercial, or industrial structures. Overall, the image portrays a bustling urban environment with a mix of residential and commercial structures."},
    {"id": 2, "image_name": "image_2.png", "caption": "In the image, you can see a large city with a densely populated urban area. The city appears to be densely populated, with numerous buildings, roads, and parks. There are also a few trees visible in the city, adding to the urban environment. The image also shows a river running through the city. The river is located in the middle of the urban area, providing a natural connection between the city and its surroundings. The presence of the river suggests that the city is connected to water resources, such as rivers, lakes, or other bodies of water. Overall, the image portrays a bustling urban environment with a mix of residential, commercial, and industrial areas."}
]

In [ ]:


vectors = embedder.encode([doc["caption"] for doc in documents])

client.upsert(
    collection_name=collection_name,
    points=[
        {
            "id": doc["id"],
            "vector": vector.tolist(),
            "payload": doc  # Store image_name and caption
        }
        for doc, vector in zip(documents, vectors)
    ]
)


UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [ ]:
query = "Show me images with a lot of vegetation."
query_vector = embedder.encode(query)

search_result = client.search(
    collection_name=collection_name,
    query_vector=query_vector.tolist(),
    limit=2
)



for hit in search_result:
    print(hit.payload)


{'id': 2, 'image_name': 'image_2.png', 'caption': 'In the image, you can see a large city with a densely populated urban area. The city appears to be densely populated, with numerous buildings, roads, and parks. There are also a few trees visible in the city, adding to the urban environment. The image also shows a river running through the city. The river is located in the middle of the urban area, providing a natural connection between the city and its surroundings. The presence of the river suggests that the city is connected to water resources, such as rivers, lakes, or other bodies of water. Overall, the image portrays a bustling urban environment with a mix of residential, commercial, and industrial areas.'}
{'id': 1, 'image_name': 'image_1.png', 'caption': 'In the image, you can see a large city with a river running through it. The city appears to be located near a body of water, such as a river, lake, or ocean. Additionally, there are numerous buildings scattered throughout the 

<ipython-input-11-cfe5f75077b2>:4: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  search_result = client.search(


In [ ]:
# Vector Search and Retrieval from Qdrant


def retrieve(query):
    query_vector = embedder.encode(query).tolist()
    search_result = client.search(
        collection_name=collection_name,
        query_vector=query_vector,
        limit=2
    )
    return search_result

query = "Show me satellite images of a city near mountains and a river."

search_result = retrieve(query)

for hit in search_result:
    print(f"Image: {hit.payload['image_name']}")
    print(f"Caption: {hit.payload['caption']}\n")


Image: image_2.png
Caption: In the image, you can see a large city with a densely populated urban area. The city appears to be densely populated, with numerous buildings, roads, and parks. There are also a few trees visible in the city, adding to the urban environment. The image also shows a river running through the city. The river is located in the middle of the urban area, providing a natural connection between the city and its surroundings. The presence of the river suggests that the city is connected to water resources, such as rivers, lakes, or other bodies of water. Overall, the image portrays a bustling urban environment with a mix of residential, commercial, and industrial areas.

Image: image_1.png
Caption: In the image, you can see a large city with a river running through it. The city appears to be located near a body of water, such as a river, lake, or ocean. Additionally, there are numerous buildings scattered throughout the city, indicating that it is a densely populated

<ipython-input-12-1642fbec6cff>:6: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  search_result = client.search(


In [ ]:
# DeepSeek Chat-Based Answering on Retrieved Captions

# Load environment variables
load_dotenv()

# Fetch the API key securely
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")

if not DEEPSEEK_API_KEY:
    raise EnvironmentError("❌ DEEPSEEK_API_KEY not found. Please set it in the .env file.")

# Setup request headers
headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {DEEPSEEK_API_KEY}"
}

# Example: Build the prompt from retrieved Qdrant results
context = "\n\n".join([hit.payload['caption'] for hit in search_result])
prompt = f"""Based only on the following satellite image captions, answer the user's question.

Context:
{context}

Question:
{query}

Answer:"""

# Query DeepSeek
def query_deepseek(prompt):
    data = {
        "model": "deepseek-chat",
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt}
        ],
        "temperature": 0.3
    }

    response = requests.post(
        "https://api.deepseek.com/v1/chat/completions",
        headers=headers,
        json=data
    )

    if response.ok:
        result = response.json()
        return result['choices'][0]['message']['content']
    else:
        print(response.json())
        raise Exception(f"Error {response.status_code}: {response.text}")


In [ ]:
captions = "\n\n".join([hit.payload['caption'] for hit in search_result])

prompt = f"""
You are a helpful assistant. Based only on the following satellite image descriptions, answer the user's question.

Context:
{captions}

Question:
What is there in the image?
"""



In [ ]:
answer = query_deepseek(prompt)
print(answer)


Based on the satellite image descriptions provided, the image shows:

1. A large, densely populated urban city with numerous buildings (residential, commercial, and industrial).
2. A river running through the middle of the city, indicating a connection to water resources.
3. Roads, parks, and a few trees scattered throughout the urban environment.
4. A bustling mix of urban structures and natural elements (the river), suggesting a lively cityscape. 

The presence of the river is a prominent feature, highlighting the city's proximity to a significant water source. Overall, the image depicts a vibrant, developed urban area with both man-made infrastructure and natural features.


In [ ]:
import nltk
from nltk.translate.bleu_score import sentence_bleu

# Define a reference answer (this would ideally come from human evaluation)
reference_answer = """
In the image, there are several key objects that can be identified. Some of these objects include a city, a mountain range, and a river. The city is located in the middle of the mountain range. The mountain range is located on the left side of the image and the river is on the right side. The river appears to be a large body of water flowing through the mountains.
In the image, you can see a large city with many buildings, roads, and other structures. The city appears to be a mix of residential and commercial areas. There is a river running through the middle of the city, which adds to the urban atmosphere. Additionally, there is<phrase> a river</phrase> on the left side of the image and another river on the right side.
"""

# Tokenize the answers for BLEU calculation
hypothesis = answer.split()
reference = [reference_answer.split()]  # Note: wrapped in list for multiple references

# Calculate BLEU score
BLEUscore = sentence_bleu(reference, hypothesis)
print(f"\nBLEU Score: {BLEUscore:.4f}")

# Optional: Calculate individual n-gram scores
for n in range(1, 5):
    weights = [1/n if i < n else 0 for i in range(4)]  # Weights for n-gram
    score = sentence_bleu(reference, hypothesis, weights=weights)
    print(f"{n}-gram BLEU: {score:.4f}")


BLEU Score: 0.0910
1-gram BLEU: 0.2794
2-gram BLEU: 0.1907
3-gram BLEU: 0.1230
4-gram BLEU: 0.0910
